## 섹션 0-1: 환경 감지 및 경로 설정

In [1]:
import os, sys

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/structure-stability-data/data'
    SRC_DIR  = '/kaggle/input/structure-stability-src/src'
    OUT_DIR  = '/kaggle/working/outputs'
    IMG_SIZE = 384
    NUM_WORKERS = 2
else:
    DATA_DIR = '../data'
    SRC_DIR  = '../src'
    OUT_DIR  = '../outputs'
    IMG_SIZE = 224
    NUM_WORKERS = 0

sys.path.append(SRC_DIR)

FIG_DIR = f"{OUT_DIR}/../reports/figures" if not IS_KAGGLE else f"{OUT_DIR}/figures"
os.makedirs(f'{OUT_DIR}/models', exist_ok=True)
os.makedirs(f'{OUT_DIR}/logs', exist_ok=True)
os.makedirs(f'{OUT_DIR}/submissions', exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"환경: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"IMG_SIZE: {IMG_SIZE}")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")

환경: Local
IMG_SIZE: 224
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design


## 섹션 0-2: import 및 seed 고정

In [2]:
import random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision.models import ResNet18_Weights, EfficientNet_B0_Weights
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, accuracy_score
from dataset import build_datasets, build_test_dataset
from augmentation import get_train_transform, get_val_transform

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


## 섹션 0-3: EarlyStopping

In [3]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False

## 섹션 0-4: train_one_epoch / evaluate

In [4]:
def train_one_epoch(model, loader, optimizer, criterion, device, use_physics=False):
    model.train()
    total_loss = 0.0
    for batch in loader:
        front = batch['front'].to(device)
        top = batch['top'].to(device)
        labels = batch['label'].view(-1, 1).to(device)
        optimizer.zero_grad()
        if use_physics:
            logits = model(front, top, batch['physics'].to(device))
        else:
            logits = model(front, top)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * front.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device, use_physics=False):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            front = batch['front'].to(device)
            top = batch['top'].to(device)
            labels = batch['label'].cpu().numpy()
            if use_physics:
                logits = model(front, top, batch['physics'].to(device))
            else:
                logits = model(front, top)
            probs = np.clip(torch.sigmoid(logits).cpu().numpy().flatten(), 1e-7, 1-1e-7)
            preds.extend(probs)
            targets.extend(labels)
    preds, targets = np.array(preds), np.array(targets)
    return log_loss(targets, preds, labels=[0,1]), accuracy_score(targets, (preds>=0.5).astype(int))

## 섹션 0-5: run_experiment

In [5]:
def run_experiment(config):
    exp_id = config['exp_id']
    set_seed(config['random_state'])

    with open(f"{config['out_dir']}/logs/exp{exp_id}_config.json", 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    train_ds, val_ds = build_datasets(config)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,
                              num_workers=config['num_workers'], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'], shuffle=False,
                              num_workers=config['num_workers'], pin_memory=True)

    model = MultiViewNet(
        backbone_name=config['backbone'],
        use_physics=config['use_physics'],
        physics_dim=config.get('physics_dim', 6),
        fusion_mode=config['fusion_mode'],
        img_size=config['img_size']
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    es = EarlyStopping(patience=config['early_stopping_patience'])

    logs = []
    best_val_logloss = float('inf')
    model_path = f"{config['out_dir']}/models/{config['model_name']}_v{config['model_version']}_best.pth"

    print(f"=== EXP-{exp_id} 시작 === Train: {len(train_ds)} | Val: {len(val_ds)}")

    for epoch in range(config['epochs']):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, config['use_physics'])
        val_logloss, val_acc = evaluate(model, val_loader, DEVICE, config['use_physics'])
        is_best = es.step(val_logloss)

        if is_best:
            best_val_logloss = val_logloss
            torch.save(model.state_dict(), model_path)

        logs.append({'epoch': epoch+1, 'train_loss': round(train_loss,6),
                     'val_logloss': round(val_logloss,6), 'val_acc': round(val_acc,4)})
        print(f"Epoch {epoch+1:02d}/{config['epochs']} | Train: {train_loss:.4f} | "
              f"Val LogLoss: {val_logloss:.4f} | Acc: {val_acc:.4f} {'✅' if is_best else ''}")

        if es.should_stop:
            print(f"Early Stopping at Epoch {epoch+1}")
            break

    log_df = pd.DataFrame(logs)
    log_df.to_csv(f"{config['out_dir']}/logs/exp{exp_id}_log.csv", index=False)
    best_ep = int(log_df.loc[log_df['val_logloss'].idxmin(), 'epoch'])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(log_df['epoch'], log_df['train_loss'], color='blue', label='Train Loss')
    axes[0].set_title(f'EXP-{exp_id} Train Loss')
    axes[1].plot(log_df['epoch'], log_df['val_logloss'], color='orange', label='Val LogLoss')
    axes[1].plot(best_ep, log_df['val_logloss'].min(), 'ro', label='Best')
    axes[1].set_title(f'EXP-{exp_id} Val LogLoss')
    for ax in axes: ax.legend()
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/exp{exp_id}_loss_curve.png", dpi=150)
    plt.show()

    print(f"Best Val LogLoss: {best_val_logloss:.4f} (Epoch {best_ep})")
    return {'exp_id': exp_id, 'best_val_logloss': best_val_logloss, 'best_epoch': best_ep, 'model_path': model_path}

## 섹션 1-1: MultiViewNet

In [6]:
class MultiViewNet(nn.Module):
    """
    학술 근거:
    🟢 Su et al.(2015) MVCNN — Multi-View Late Fusion
    🟢 Guo et al.(2017) — FC 벡터 추출 구조 (SHAP 연결용)
    🟡 Lerer et al.(2016) PhysNet — 물리 피처 결합
    """
    def __init__(self, backbone_name='resnet18', use_physics=False,
                 physics_dim=6, fusion_mode='concat', img_size=224):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.use_physics = use_physics
        self.front_encoder, feat_dim = self._build_backbone(backbone_name)
        self.top_encoder,   _        = self._build_backbone(backbone_name)
        self.feature_dim = feat_dim
        fc_in = feat_dim * 2 if fusion_mode == 'concat' else feat_dim * 3
        if use_physics:
            fc_in += physics_dim
        self.fc_hidden = nn.Sequential(nn.Linear(fc_in, 256), nn.ReLU(), nn.Dropout(0.3))
        self.fc_out = nn.Linear(256, 1)

    def _build_backbone(self, name):
        if name == 'resnet18':
            net = models.resnet18(weights=ResNet18_Weights.DEFAULT)
            return nn.Sequential(*list(net.children())[:-1]), 512
        elif name == 'efficientnet_b0':
            net = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
            return nn.Sequential(net.features, nn.AdaptiveAvgPool2d(1)), 1280
        raise ValueError(f"Unknown backbone: {name}")

    def extract_features(self, front, top, physics=None):
        f1 = self.front_encoder(front).flatten(1)
        f2 = self.top_encoder(top).flatten(1)
        fused = torch.cat([f1, f2], dim=1) if self.fusion_mode == 'concat' else torch.cat([f1, f2, f1-f2], dim=1)
        if self.use_physics and physics is not None:
            fused = torch.cat([fused, physics], dim=1)
        return fused

    def forward(self, front, top, physics=None):
        return self.fc_out(self.fc_hidden(self.extract_features(front, top, physics)))

## 섹션 1-2: 구조 확인

In [7]:
test_model = MultiViewNet(backbone_name='resnet18', fusion_mode='concat', img_size=IMG_SIZE).to(DEVICE)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out = test_model(dummy, dummy)
print(f"출력 shape: {out.shape}")
print(f"총 파라미터: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model, dummy

출력 shape: torch.Size([2, 1])
총 파라미터: 22,615,681


## 섹션 2: EXP-001 (HIGH 증강)

In [ ]:
config_exp001 = {
    "exp_id": "001", "model_version": "2", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "concat",
    "use_physics": False, "use_physics_features": False, "physics_dim": 6,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "imagenet", "dev_split_ratio": 0.0, "img_size": IMG_SIZE,
    "aug_params": {
        "brightness_p": 0.7,   # 🟢 Hendrycks(2020)
        "gamma_p": 0.5,        # 🟡 조명 비선형 변화
        "hsv_p": 0.5,          # 🟡 Shorten(2019)
        "shift_scale_p": 0.4,  # 🟢 Krizhevsky(2012)
        "perspective_p": 0.4,  # 🔴 휴리스틱
        "flip_p": 0.3          # 🔴 휴리스틱
    },
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 5, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp001 = run_experiment(config_exp001)

## 섹션 3: EXP-002 (Dev 학습 포함)

In [ ]:
config_exp002 = {**config_exp001, "exp_id": "002", "model_version": "3", "dev_split_ratio": 0.8}
result_exp002 = run_experiment(config_exp002)
print("⚠️ Val 샘플 20개 — EXP-001(Val 100개)과 직접 비교 주의")

## 섹션 4: EXP-004 (Custom 정규화)

In [ ]:
config_exp004 = {**config_exp001, "exp_id": "004", "model_version": "4", "norm_version": "custom"}
result_exp004 = run_experiment(config_exp004)

## 섹션 5: EXP-005 (EfficientNet-B0)

In [ ]:
# VRAM 사전 확인
test_eff = MultiViewNet(backbone_name='efficientnet_b0', fusion_mode='concat', img_size=IMG_SIZE).to(DEVICE)
dummy = torch.randn(16, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
_ = test_eff(dummy, dummy)
if torch.cuda.is_available():
    print(f"VRAM 사용량: {torch.cuda.memory_allocated()/(1024**2):.1f} MB")
del test_eff, dummy; torch.cuda.empty_cache()

config_exp005 = {**config_exp001, "exp_id": "005", "model_version": "5",
                 "model_name": "efficientnet", "backbone": "efficientnet_b0"}
result_exp005 = run_experiment(config_exp005)

## 섹션 6: EXP-006 (물리 피처)

In [ ]:
config_exp006 = {**config_exp001, "exp_id": "006", "model_version": "6",
                 "use_physics": True, "use_physics_features": True, "physics_dim": 6}
result_exp006 = run_experiment(config_exp006)

## 섹션 7: EXP-007 (Fusion 구조 개선)

In [ ]:
config_exp007 = {**config_exp001, "exp_id": "007", "model_version": "7", "fusion_mode": "diff_concat"}
result_exp007 = run_experiment(config_exp007)

## 섹션 8: 실험 결과 비교표

In [ ]:
results = [
    {"exp_id": "baseline", "model": "resnet18", "변경사항": "없음(기준점)",
     "best_val_logloss": 1.5369, "best_epoch": 1},
    {**result_exp001, "model": "resnet18", "변경사항": "HIGH 증강"},
    {**result_exp002, "model": "resnet18", "변경사항": "Dev 학습 포함"},
    {**result_exp004, "model": "resnet18", "변경사항": "Custom 정규화"},
    {**result_exp005, "model": "efficientnet_b0", "변경사항": "백본 교체"},
    {**result_exp006, "model": "resnet18", "변경사항": "물리 피처"},
    {**result_exp007, "model": "resnet18", "변경사항": "diff_concat Fusion"},
]
result_df = pd.DataFrame(results)[['exp_id','model','변경사항','best_val_logloss','best_epoch']].sort_values('best_val_logloss')
print(result_df.to_string(index=False))
result_df.to_csv(f"{OUT_DIR}/logs/experiment_summary.csv", index=False)